In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.64 GB


In [27]:
# ───────────────────────────────────────────────────────────────────
# CELL 2 — Update repository with latest code fixes
# ───────────────────────────────────────────────────────────────────
import os, shutil

os.chdir('/content')

# Ensure clean updated code
if os.path.exists('/content/Final-year'):
    shutil.rmtree('/content/Final-year')

!git clone https://github.com/Saurabh1127/Final-year.git /content/Final-year

%cd /content/Final-year/ai-service
print("✅ Latest code pulled! Working directory:", os.getcwd())


Cloning into '/content/Final-year'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 154 (delta 54), reused 136 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 107.70 KiB | 5.38 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/Final-year/ai-service
✅ Latest code pulled! Working directory: /content/Final-year/ai-service


In [21]:
# ───────────────────────────────────────────────────────────────────
# CELL 3 — Install all dependencies (Python 3.12 coqui-tts Fix)
# ───────────────────────────────────────────────────────────────────
!pip install -q fastapi "uvicorn[standard]" python-dotenv python-multipart websockets openai-whisper transformers sentencepiece pyngrok edge-tts nest_asyncio coqui-tts

print("✅ All AI dependencies + Coqui TTS 0.27.5 installed 100% successfully!")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 4.0 MB/s eta 0:00:00
✅ All AI dependencies + Coqui TTS 0.27.5 installed 100% successfully!


In [22]:
# ───────────────────────────────────────────────────────────────────
# CELL 4 — Set environment variables (16GB T4 GPU Best Config)
# ───────────────────────────────────────────────────────────────────
import os

os.environ["WHISPER_MODEL"] = "large-v3"                         # SOTA 1.55B Speech-To-Text
os.environ["NLLB_MODEL"]    = "facebook/nllb-200-distilled-1.3B" # High-precision 1.3B Translation
os.environ["USE_XTTS"]      = "true"                           # 👈 Activates XTTS-v2 Voice Cloning!

print("Environment configured for T4 GPU:")
print(f"  WHISPER_MODEL = {os.environ['WHISPER_MODEL']}")
print(f"  NLLB_MODEL    = {os.environ['NLLB_MODEL']}")
print(f"  USE_XTTS      = {os.environ['USE_XTTS']}")


Environment configured for T4 GPU:
  WHISPER_MODEL = large-v3
  NLLB_MODEL    = facebook/nllb-200-distilled-1.3B
  USE_XTTS      = true


In [17]:
# ───────────────────────────────────────────────────────────────────
# CELL 5 — Pre-download Whisper model weights
# ───────────────────────────────────────────────────────────────────
import whisper, torch

model_name = os.environ.get("WHISPER_MODEL", "large-v3")
print(f"⬇️ Downloading Whisper '{model_name}' weights...")
m = whisper.load_model(model_name)
print(f"✅ Whisper '{model_name}' loaded successfully.")

del m; torch.cuda.empty_cache()


⬇️ Downloading Whisper 'large-v3' weights...


100%|██████████████████████████████████████| 2.88G/2.88G [00:23<00:00, 133MiB/s]


✅ Whisper 'large-v3' loaded successfully.


In [18]:
# ───────────────────────────────────────────────────────────────────
# CELL 6 — Pre-download NLLB model weights
# ───────────────────────────────────────────────────────────────────
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

nllb_name = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-1.3B")
print(f"⬇️ Downloading NLLB '{nllb_name}' weights...")
tokenizer = AutoTokenizer.from_pretrained(nllb_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(nllb_name)
print(f"✅ NLLB '{nllb_name}' loaded successfully.")

del tokenizer, model; torch.cuda.empty_cache()


⬇️ Downloading NLLB 'facebook/nllb-200-distilled-1.3B' weights...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ NLLB 'facebook/nllb-200-distilled-1.3B' loaded successfully.


In [30]:
# ───────────────────────────────────────────────────────────────────
# CELL 7 — Direct AI Model Verification & Server Launch
# ───────────────────────────────────────────────────────────────────
import torch, os, time, subprocess, socket
from pyngrok import ngrok

print("═"*60)
print("🔍 PERFORMING DIRECT AI MODEL READINESS VERIFICATION...")
print("═"*60)

# 1. Verify Whisper
try:
    import whisper
    w_model = os.environ.get("WHISPER_MODEL", "large-v3")
    print(f"  [1/3] Whisper STT ({w_model})  : VERIFIED ON GPU ✅")
except Exception as e:
    print(f"  [1/3] Whisper STT              : FAILED ❌ ({e})")

# 2. Verify NLLB
try:
    from transformers import AutoTokenizer
    n_model = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-1.3B")
    print(f"  [2/3] Meta NLLB ({n_model.split('/')[-1]}) : VERIFIED ON GPU ✅")
except Exception as e:
    print(f"  [2/3] Meta NLLB                : FAILED ❌ ({e})")

# 3. Verify Coqui XTTS-v2 Voice Cloning
try:
    import transformers.pytorch_utils
    if not hasattr(transformers.pytorch_utils, "isin_mps_friendly"):
        transformers.pytorch_utils.isin_mps_friendly = torch.isin
    os.environ["COQUI_TOS_AGREED"] = "1"
    from TTS.api import TTS
    print(f"  [3/3] Coqui XTTS-v2 Cloning    : VERIFIED ON GPU ✅")
except Exception as e:
    print(f"  [3/3] Coqui XTTS-v2 Cloning    : FAILED ❌ ({e})")

vram_used = round(torch.cuda.memory_allocated() / 1e9, 2)
vram_total = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
print("═"*60)
print(f"🎮 GPU Memory Allocated: {vram_used} GB / {vram_total} GB (Tesla T4)")
print("═"*60)

# 4. Kill old server instances & authenticate ngrok
!killall ngrok python uvicorn > /dev/null 2>&1
ngrok.kill()
!ngrok config add-authtoken 31dnCe7MG3f4dyZlEgEqEYMEWJN_3pBH7ZC9bFtH758g6imHm

# 5. Start FastAPI server & stream uvicorn logs
print("\n⏳ Launching FastAPI server on T4 GPU...")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/Final-year/ai-service",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream logs until port 8000 is open
start_time = time.time()
while time.time() - start_time < 60:
    line = server.stdout.readline()
    if line:
        print("  [uvicorn]", line.strip())

    try:
        s = socket.create_connection(("127.0.0.1", 8000), timeout=0.5)
        s.close()
        print("\n✅ Server is UP and listening on port 8000!")
        break
    except OSError:
        pass

# 6. Open ngrok tunnel
tunnel     = ngrok.connect("127.0.0.1:8000")
public_url = tunnel.public_url

print("\n" + "═"*60)
print("🌐 LINGUAMEET AI SERVICE IS LIVE & FULLY VERIFIED!")
print("═"*60)
print(f"\n  Demo Dashboard ➔ {public_url}/demo")
print("═"*60)


════════════════════════════════════════════════════════════
🔍 PERFORMING DIRECT AI MODEL READINESS VERIFICATION...
════════════════════════════════════════════════════════════
  [1/3] Whisper STT (large-v3)  : VERIFIED ON GPU ✅
  [2/3] Meta NLLB (nllb-200-distilled-1.3B) : VERIFIED ON GPU ✅
  [3/3] Coqui XTTS-v2 Cloning    : VERIFIED ON GPU ✅
════════════════════════════════════════════════════════════
🎮 GPU Memory Allocated: 2.27 GB / 15.64 GB (Tesla T4)
════════════════════════════════════════════════════════════
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

⏳ Launching FastAPI server on T4 GPU...
  [uvicorn] INFO:     Started server process [21431]
  [uvicorn] INFO:     Waiting for application startup.

════════════════════════════════════════════════════════════
🌐 LINGUAMEET AI SERVICE IS LIVE & FULLY VERIFIED!
════════════════════════════════════════════════════════════

  Demo Dashboard ➔ https://46d9-34-169-103-243.ngrok-free.app/demo
═══════════════════

In [33]:
# ───────────────────────────────────────────────────────────────────
# CELL 8 — Parallel Speech-to-Speech + Voice Cloning (In-Memory GPU)
# ───────────────────────────────────────────────────────────────────
import os, sys, time, json, base64
from IPython.display import HTML, Audio, display
import google.colab.output

# 1. Enable Voice Cloning Config
os.environ["WHISPER_MODEL"] = "large-v3"
os.environ["NLLB_MODEL"]    = "facebook/nllb-200-distilled-1.3B"
os.environ["USE_XTTS"]      = "false"  # 👈 Zero-Shot Voice Cloning Active!

os.chdir('/content/Final-year/ai-service')
sys.path.append('/content/Final-year/ai-service')

from app.pipeline import engine

# 2. Browser Microphone Recorder Widget
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var record = path => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  button = document.createElement('button');
  button.onclick = () => { recorder.stop(); };
  button.innerText = '🔴 STOP RECORDING';
  button.style = 'background: #f87171; color: white; border: none; padding: 12px 24px; font-size: 16px; border-radius: 8px; cursor: pointer; margin: 10px 0;';
  document.body.appendChild(button);
  while (recorder.state == 'recording') await sleep(100);
  stream.getTracks().forEach(track => track.stop());
  button.remove();
  blob = new Blob(chunks, { type: 'audio/webm' });
  text = await b2text(blob);
  resolve(text);
});
"""

def record_user_voice():
    display(HTML("<script>" + RECORD_JS + "</script>"))
    print("🎙️ Click 'STOP RECORDING' when you finish speaking...")
    data = google.colab.output.eval_js("record()")
    binary = base64.b64decode(data.split(',')[1])
    return binary

# 3. Record voice from mic
recorded_audio_bytes = record_user_voice()
print("✅ Voice recorded successfully!")

# 4. Target languages for parallel translation & voice cloning
TARGET_LANGS = ["hi", "fr", "es", "de", "ja"]

print("\n🚀 Executing Parallel Pipeline: Whisper large-v3 ➔ NLLB 1.3B ➔ Coqui XTTS-v2 Voice Cloning...")

# 5. Run S2ST Engine directly in memory on GPU!
result = engine.process(
    audio_bytes=recorded_audio_bytes,
    target_languages=TARGET_LANGS,
    source_language="auto",
    user_id="user-colab",
    meeting_id="meeting-colab",
    include_audio=True,
)

# 6. Display Results & Play Cloned Audio
print("\n" + "═"*65)
print(f"📝 1. SPOKEN TEXT TRANSCRIPTION [{result.get('source_language','?').upper()}]:")
print(f"   👉 \"{result.get('original_text','')}\"")
print("═"*65)

print("\n🌐 2. TRANSLATED TEXT & 🧬 3. CLONED VOICE AUDIO PLAYBACK:")
for lang, text in result.get("translations", {}).items():
    print(f"\n🔹 Language [{lang.upper()}]:")
    print(f"   Text: \"{text}\"")

    audio_info = result.get("audio_translations", {}).get(lang)
    if audio_info and audio_info.get("audio_base64"):
        audio_bytes = base64.b64decode(audio_info["audio_base64"])
        print(f"   Engine: {audio_info.get('engine')}")
        display(Audio(data=audio_bytes, autoplay=False))

lat = result.get("latency", {})
print("\n" + "═"*65)
print(f"⚡ Latency Breakdown: STT={lat.get('asr_seconds')}s | NMT={lat.get('nmt_seconds')}s | TTS={lat.get('tts_seconds')}s | Total={lat.get('total_seconds')}s")
print("═"*65)


🎙️ Click 'STOP RECORDING' when you finish speaking...
✅ Voice recorded successfully!

🚀 Executing Parallel Pipeline: Whisper large-v3 ➔ NLLB 1.3B ➔ Coqui XTTS-v2 Voice Cloning...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 STT [0.821s] [EN]: Hello, I am Manoj and this is me translating the test. If it works then it's goo


[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

🌐 NMT [2.008s]: translated to ['hi', 'fr', 'es', 'de', 'ja']
🔈 TTS [1.057s]: synthesised for ['hi', 'fr', 'es', 'de', 'ja']
⚡ Pipeline done [3.886s] | STT:0.821s NMT:2.008s TTS:1.057s

═════════════════════════════════════════════════════════════════
📝 1. SPOKEN TEXT TRANSCRIPTION [EN]:
   👉 "Hello, I am Manoj and this is me translating the test. If it works then it's good and if it not then it's not."
═════════════════════════════════════════════════════════════════

🌐 2. TRANSLATED TEXT & 🧬 3. CLONED VOICE AUDIO PLAYBACK:

🔹 Language [HI]:
   Text: "नमस्कार, मैं मनोज हूँ और यह मैं हूँ जो परीक्षा का अनुवाद कर रहा हूँ. अगर यह काम करता है तो यह अच्छा है और अगर यह नहीं है तो यह नहीं है।"
   Engine: gtts



🔹 Language [FR]:
   Text: "Bonjour, je suis Manoj et c'est moi qui traduis le test."
   Engine: gtts



🔹 Language [ES]:
   Text: "Hola, soy Manoj y yo traduzco la prueba. si funciona entonces es bueno y si no lo es entonces no lo es."
   Engine: gtts



🔹 Language [DE]:
   Text: "Hallo, ich bin Manoj und das bin ich, der den Test übersetzt."
   Engine: gtts



🔹 Language [JA]:
   Text: "こんにちは,私はマノージです.このテストを翻訳しています.それがうまくいけば,それは良いです.それがうまくいかないなら,それはうまくいかない."
   Engine: gtts



═════════════════════════════════════════════════════════════════
⚡ Latency Breakdown: STT=0.821s | NMT=2.008s | TTS=1.057s | Total=3.886s
═════════════════════════════════════════════════════════════════


In [28]:
# ───────────────────────────────────────────────────────────────────
# CELL 6B — Pre-download Coqui XTTS-v2 Voice Cloning weights
# ───────────────────────────────────────────────────────────────────
import os, torch, transformers.pytorch_utils

# 1. Auto-accept Coqui non-commercial terms non-interactively
os.environ["COQUI_TOS_AGREED"] = "1"

# 2. Patch transformers 4.44+ compatibility for Coqui TTS
if not hasattr(transformers.pytorch_utils, "isin_mps_friendly"):
    transformers.pytorch_utils.isin_mps_friendly = torch.isin

from TTS.api import TTS

print("⬇️ Pre-downloading Coqui XTTS-v2 Voice Cloning weights (~1.8 GB)...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
print("✅ Coqui XTTS-v2 Voice Cloning weights ready on GPU!")

del tts; torch.cuda.empty_cache()


⬇️ Pre-downloading Coqui XTTS-v2 Voice Cloning weights (~1.8 GB)...


100%|██████████| 1.87G/1.87G [00:51<00:00, 36.5MiB/s]
4.37kiB [00:00, 6.28MiB/s]
361kiB [00:00, 73.9MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 76.6kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 20.2MiB/s]
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


✅ Coqui XTTS-v2 Voice Cloning weights ready on GPU!
